<a href="https://colab.research.google.com/github/sharaffatima/bachelor-thesis/blob/main/notebooks/01_preprocessing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 1. Drive
connect to Drive to access PTBXL ECG Dataset.

In [11]:
from google.colab import drive
drive.mount("/content/drive/")

Drive already mounted at /content/drive/; to attempt to forcibly remount, call drive.mount("/content/drive/", force_remount=True).


In [13]:
# create a temporarly copy of data in colab for faster preprocessing
!cp -r "/content/drive/MyDrive/ECG data/raw/ptbxl" "/content/ptbxl"

In [2]:
! pip install wfdb
! pip install tqdm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 163.9/163.9 kB 14.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 75.3 MB/s eta 0:00:00
  Attempting uninstall: pandas
    Found existing installation: pandas 2.2.2
    Uninstalling pandas-2.2.2:
      Successfully uninstalled pandas-2.2.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 3.0.3 which is incompatible.
db-dtypes 1.5.1 requires pandas<3.0.0,>=1.5.3, but you have pandas 3.0.3 which is incompatible.
dask-cudf-cu12 26.2.1 requires pandas<2.4.0,>=2.0, but you have pandas 3.0.3 which is incompatible.
cudf-cu12 26.2.1 requires pandas<2.4.0,>=2.0, but you have pandas 3.0.3 which is incompatible.
bqplot 0.12.45 requires pandas<3.0.0,>=1.0.0, but 

In [14]:
import os
import sys
from pathlib import Path
import ast
import wfdb
import numpy as np
import pandas as pd
from tqdm import tqdm

# 2. Paths to dataset

In [15]:
DATA_DIR = Path("/content/drive/MyDrive/ECG data")

RAW_DIR = Path("/content/ptbxl")  # raw data is stored temporarly in colab for faster preprocessing
PROCESSED_DIR = DATA_DIR / "processed"
RESULTS_DIR = DATA_DIR / "results"

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print("DATA_DIR exists:", DATA_DIR.exists())
print("RAW_DIR exists:", RAW_DIR.exists())
print("ptbxl_database exists:", (RAW_DIR / "ptbxl_database.csv").exists())
print("scp_statements exists:", (RAW_DIR / "scp_statements.csv").exists())
print("records100 exists:", (RAW_DIR / "records100").exists())

print("PROCESSED_DIR created", PROCESSED_DIR.exists())
print("RESULTS_DIR created", RESULTS_DIR.exists())

DATA_DIR exists: True
RAW_DIR exists: True
ptbxl_database exists: True
scp_statements exists: True
records100 exists: True
PROCESSED_DIR created True
RESULTS_DIR created True


# 3. Load Metadata of PTB XL

In [16]:
df = pd.read_csv(os.path.join(RAW_DIR, "ptbxl_database.csv"))

df["scp_codes"] = df["scp_codes"].apply(ast.literal_eval)

# 4. Labels

0 = normal

1 = anomaly

In [17]:
def is_normal(scp_codes):
    return "NORM" in scp_codes.keys()

df["label"] = df["scp_codes"].apply(lambda codes: 0 if is_normal(codes) else 1)

# 5. Split
Now I create the Split based on the official PTB XL recommended strat_folds:

8 Training, 1 Validation, 1 Testing

In [18]:
train_df = df[df["strat_fold"].between(1, 8)]
val_df = df[df["strat_fold"] == 9]
test_df = df[df["strat_fold"] == 10]

print("Train:")
print(train_df["label"].value_counts())

print("\nValidation:")
print(val_df["label"].value_counts())

print("\nTest:")
print(test_df["label"].value_counts())

Train:
label
1    9822
0    7596
Name: count, dtype: int64

Validation:
label
1    1228
0     955
Name: count, dtype: int64

Test:
label
1    1235
0     963
Name: count, dtype: int64


# 6. Load ECG records

In [19]:
def load_ecg_records(dataframe, base_path):
    signals = []
    labels = []

    for _, row in tqdm(dataframe.iterrows(), total=len(dataframe)):
        record_path = os.path.join(base_path, row["filename_lr"])
        record = wfdb.rdrecord(record_path)

        signal = record.p_signal.astype(np.float32)

        # Expected shape: [1000, 12]
        if signal.shape != (1000, 12):
            continue

        signals.append(signal)
        labels.append(row["label"])

    X = np.stack(signals)
    y = np.array(labels, dtype=np.int64)

    return X, y

In [20]:
X_train, y_train = load_ecg_records(train_df, RAW_DIR)
X_val, y_val = load_ecg_records(val_df, RAW_DIR)
X_test, y_test = load_ecg_records(test_df, RAW_DIR)

100%|██████████| 2198/2198 [00:20<00:00, 108.85it/s]


# 7. Check shapes



In [21]:
print(X_train.shape, y_train.shape)
print(X_val.shape, y_val.shape)
print(X_test.shape, y_test.shape)

(17418, 1000, 12) (17418,)
(2183, 1000, 12) (2183,)
(2198, 1000, 12) (2198,)


# 8. Save processed arrays

In [22]:
np.save(os.path.join(PROCESSED_DIR, "X_train.npy"), X_train)
np.save(os.path.join(PROCESSED_DIR, "y_train.npy"), y_train)

np.save(os.path.join(PROCESSED_DIR, "X_val.npy"), X_val)
np.save(os.path.join(PROCESSED_DIR, "y_val.npy"), y_val)

np.save(os.path.join(PROCESSED_DIR, "X_test.npy"), X_test)
np.save(os.path.join(PROCESSED_DIR, "y_test.npy"), y_test)